# COMP6008 Smart Irrigation RL — Shared Group Notebook

**Location:** Kyabram, Victoria  
**Algorithms:** DDPG, TD3 and SAC  
**Status:** Shared environment and experiment framework complete; agent implementations and experimental results remain to be completed.

This notebook is the common starting point for all three agents. It covers the project objectives, fixed MDP, validated simulator, reproducibility controls, benchmarks, tuning rules, evaluation metrics, robustness tests and reporting structure. Each member should copy it and edit only their assigned agent section until integration.


## 0. Project-objective map

| Project objective | Where it is addressed in this notebook |
|---|---|
| 1. Explain the RL concepts and why they apply | Scientific rationale, return, exploration and actor–critic discussion below |
| 2. Formulate the environment and examine MDP assumptions | Fixed MDP, 18-value observation, transition equations, reward, partial-observability limitations and Gymnasium implementation |
| 3. Implement and tune three suitable agents | Separate DDPG, TD3 and SAC work areas, shared interface, fixed grids and validation-only selection function |
| 4. Evaluate performance, stability and generalisation | Physical metrics, learning curves, seed summaries, held-out seasons, stability/sample-efficiency functions and robustness suite |
| 5. Identify problems and explain their causes | Physics/information-timing tests, terminal checks and a structured failure log |
| 6. Document and communicate findings | Evidence–reasoning–limitations prompts, references, result tables and interpretation checklist |
| 7. Plan and share the work fairly | Agent ownership plus environment, infrastructure and evaluation responsibilities, with integration review |

The detailed scientific justification remains in [`PROJECT_OBJECTIVES.md`](../docs/PROJECT_OBJECTIVES.md). The numerical source of truth is [`FIXED_EXPERIMENT_SETTINGS.md`](../docs/FIXED_EXPERIMENT_SETTINGS.md).


## 1. Group allocation and parallel workflow

Agent owners are confirmed below; agree on shared responsibilities and review partners before making the remaining agent copies. Every member owns one agent and one shared responsibility.

| Responsibility | Group member | Branch/notebook | Review partner | Status |
|---|---|---|---|---|
| DDPG implementation, pseudocode and analysis | Raina Shah | `01_ddpg.ipynb` | TODO | Not started |
| TD3 implementation, pseudocode and analysis | Arnav Dangmali | `02_td3.ipynb` | TODO | Not started |
| SAC implementation, pseudocode and analysis | Malaika Noor | `03_sac.ipynb` | TODO | Not started |
| Environment integration and validation | TODO | Shared | TODO | Framework ready |
| Training/data infrastructure | TODO | Shared | TODO | Framework ready |
| Evaluation, plots and final integration | TODO | Shared | TODO | Framework ready |

Rules:

1. Do not independently alter the environment, data splits, reward or shared budgets in an agent notebook.
2. Record proposed changes and obtain group agreement before final testing.
3. Merge early using short smoke runs; do not wait for full training.
4. Reassess workload after the first working agent versions and redistribute shared work if necessary.
5. Every member must be able to explain the environment and all three algorithms.
6. Each member will later propose one improvement tied to an observed limitation and one separate future direction with supporting literature.

Save each agent’s results, checkpoints and visualisations in its own directory: `outputs/ddpg/`, `outputs/td3/` or `outputs/sac/`. Use `agent_output_dir("td3")` or `agent_output_dir("sac")` in the new agent notebooks.


## 2. Scientific rationale and practical-code provenance

Irrigation is sequential because today's irrigation changes tomorrow's soil-water state. The agent should therefore maximise a seasonal return rather than minimise only today's water use:

$$G_t=\sum_{k=0}^{T-t-1}\gamma^kR_{t+k+1}.$$

The policy selects irrigation, while the critic/value function estimates longer-term consequences [1, Secs. 3.3–3.5]. Exploration occurs only in the simulator; random actions are not proposed for a real field. A threshold controller is retained because the project must test whether RL adds value rather than assume that it does.

The continuous action is the reason for selecting actor–critic methods:

| Agent | Scientific reason for inclusion |
|---|---|
| DDPG | Deterministic off-policy actor–critic baseline for continuous control, with replay and target networks [4] |
| TD3 | Tests whether twin critics, delayed policy updates and target smoothing improve on DDPG's stability [5] |
| SAC | Tests a stochastic maximum-entropy approach to exploration under changing weather [6] |

These papers justify testing the algorithms; they do not predict which one will perform best here. That conclusion must come from repeated held-out experiments [7].

### Relationship to the unit practicals

This notebook follows [`PRACTICAL_CODE_REUSE.md`](../docs/PRACTICAL_CODE_REUSE.md). We adapt:

- Practical 1's Gymnasium interaction and `terminated`/`truncated` handling;
- Practicals 4–5's TD-target and terminal-mask reasoning;
- Practical 8's replay buffer, minibatches, target networks and training-loop organisation; and
- Practical 9's actor/critic separation, MLP structure, Adam optimisation and evaluation pattern.

We do **not** copy discrete-action logic unchanged. DQN's `argmax`, action-index `gather()` and one-Q-output-per-action design are unsuitable. Practical 9's categorical policy is also unsuitable. The continuous policies, action-conditioned critics and DDPG/TD3/SAC losses must be implemented from the original papers.

Each agent section must identify: the practical used as a structural starting point, what was retained, what changed for continuous control, the matching pseudocode step and the original paper supporting the final update.


## 3. Setup

Run the installation cell once per Colab session. The final run should record the resolved package versions and hardware below.


In [ ]:
%pip install -q "gymnasium>=1.0,<2" "numpy>=1.22" "torch>=2.0" "pandas>=2.0" "matplotlib>=3.7"

In [ ]:
from __future__ import annotations

import hashlib
import json
import platform
import random
import time
import warnings
from collections import deque
from dataclasses import asdict, dataclass, replace
from datetime import date, timedelta
from itertools import product
from pathlib import Path
from typing import Any
from zipfile import ZipFile

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

RUNTIME = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "gymnasium": gym.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
RUNTIME


## 4. Data, provenance and scope

The environment uses Kyabram SILO rainfall and FAO-56 reference evapotranspiration. Soil capacity comes from the audited CSIRO Soil and Landscape Grid extraction. SILO source flags remain in the archive even though the controller uses only rainfall and ETo. Interpolated values must not be described as station observations [8].

The climate forcing is historical; soil water, irrigation, drainage and crop response are simulated. The coordinate represents a virtual field, not a verified commercial paddock. The simulator is suitable for a comparative RL study, not a field-ready irrigation recommendation.

The required archive is [`data/Irrigation_Data_Audit.zip`](../data/Irrigation_Data_Audit.zip). If Colab does not mount repository files, upload that ZIP through the Files panel.


In [ ]:
DATA_CANDIDATES = [
    Path("../data/Irrigation_Data_Audit.zip"),  # when launched from notebooks/
    Path("data/Irrigation_Data_Audit.zip"),    # when working directory is repository root
    Path("Irrigation_Data_Audit.zip"),
    Path("/content/Irrigation_Data_Audit.zip"),
    Path("../irrigation_audit/kyabram_080091_daily.json"),
    Path("irrigation_audit/kyabram_080091_daily.json"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Upload Irrigation_Data_Audit.zip and rerun this cell. "
        "The expected Colab path is /content/Irrigation_Data_Audit.zip."
    )

DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Using:", DATA_PATH.resolve())
print("SHA-256:", DATA_SHA256)

# Resolve outputs consistently from repository root, notebooks/, or standalone Colab.
PROJECT_ROOT = next((p.resolve() for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "notebooks").is_dir()), Path.cwd())
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
AGENT_OUTPUT_DIRS = {name: OUTPUT_ROOT / name for name in ("ddpg", "td3", "sac")}

def agent_output_dir(agent_name):
    if agent_name not in AGENT_OUTPUT_DIRS:
        raise ValueError("Choose ddpg, td3, or sac")
    directory = AGENT_OUTPUT_DIRS[agent_name]
    directory.mkdir(parents=True, exist_ok=True)
    return directory


## 5. Frozen shared settings and declared experimental plans

These settings must be identical across DDPG, TD3 and SAC. Agent-specific selected values remain `None` until validation-only tuning is complete. Test seasons must not influence a setting, checkpoint or reward choice.


In [ ]:
TRAIN_YEARS = tuple(range(2005, 2017))
VALIDATION_YEARS = tuple(range(2017, 2021))
TEST_YEARS = tuple(range(2021, 2025))
SEASON_SPLITS = {
    "train": TRAIN_YEARS,
    "validation": VALIDATION_YEARS,
    "test": TEST_YEARS,
    "all": TRAIN_YEARS + VALIDATION_YEARS + TEST_YEARS,
}


@dataclass(frozen=True)
class EnvironmentConfig:
    capacity_mm: float = 111.33333333333334
    initial_storage_fraction: float = 0.80
    max_irrigation_mm: float = 20.0
    irrigation_efficiency: float = 0.85
    season_length_days: int = 150
    planting_month: int = 11
    planting_day: int = 1
    planting_day_offset: int = 0
    kc_multiplier: float = 1.0
    irrigation_weight: float = 1.0
    shortage_weight: float = 10.0
    drainage_weight: float = 1.0
    rainfall_scale_mm: float = 20.0
    eto_scale_mm: float = 10.0
    history_days: int = 7

    def __post_init__(self):
        if self.capacity_mm <= 0 or self.max_irrigation_mm <= 0:
            raise ValueError("Storage capacity and irrigation limit must be positive")
        if not 0 < self.initial_storage_fraction <= 1:
            raise ValueError("Initial storage fraction must be in (0, 1]")
        if not 0 < self.irrigation_efficiency <= 1:
            raise ValueError("Irrigation efficiency must be in (0, 1]")
        if self.season_length_days != 150 or self.history_days != 7:
            raise ValueError("The fixed experiment uses 150 days and seven history days")
        if not 0 < self.kc_multiplier <= 1.1:
            raise ValueError("Kc multiplier must be in (0, 1.1]")
        if min(self.irrigation_weight, self.shortage_weight, self.drainage_weight) < 0:
            raise ValueError("Reward weights must be non-negative")


ENV_CONFIG = EnvironmentConfig()

SHARED_TRAINING = {
    "gamma": 0.995,
    "tau": 0.005,
    "replay_capacity": 100_000,
    "batch_size": 128,
    "random_exploration_steps": 1_500,
    "hidden_sizes": (128, 128),
    "optimiser": "Adam",
    "optimiser_betas": (0.9, 0.999),
    "optimiser_epsilon": 1e-8,
    "updates_per_environment_step": 1,
    "final_training_steps": 100_000,
    "tuning_steps": 30_000,
    "evaluation_interval": 5_000,
    "final_checkpoint": 100_000,
    "final_seeds": (11, 22, 33, 44, 55),
    "tuning_seeds": (101, 202),
}

AGENT_TUNING_GRIDS = {
    "ddpg": [
        {"learning_rate": lr, "exploration_std": std}
        for lr, std in product((1e-4, 3e-4), (0.1, 0.2))
    ],
    "td3": [
        {"learning_rate": lr, "exploration_std": std}
        for lr, std in product((1e-4, 3e-4), (0.1, 0.2))
    ],
    "sac": [
        {"learning_rate": lr, "alpha": alpha}
        for lr, alpha in product((1e-4, 3e-4), (0.1, 0.2))
    ],
}

AGENT_SETTINGS = {
    "ddpg": {"owner": "Raina Shah", "selected_learning_rate": None, "selected_exploration_std": None},
    "td3": {
        "owner": "Arnav Dangmali", "selected_learning_rate": None, "selected_exploration_std": None,
        "policy_delay": 2, "target_smoothing_std": 0.2, "target_noise_clip": 0.5,
    },
    "sac": {
        "owner": "Malaika Noor", "selected_learning_rate": None, "selected_alpha": None,
        "formulation": "Haarnoja et al. (2018), separate value network, fixed alpha",
        "log_std_bounds": None,  # document before SAC training
    },
}

ROBUSTNESS_CONFIGS = {
    "capacity_0.8": replace(ENV_CONFIG, capacity_mm=0.8 * ENV_CONFIG.capacity_mm),
    "capacity_1.2": replace(ENV_CONFIG, capacity_mm=1.2 * ENV_CONFIG.capacity_mm),
    "root_zone_0.6m": replace(ENV_CONFIG, capacity_mm=72.0),
    "efficiency_0.75": replace(ENV_CONFIG, irrigation_efficiency=0.75),
    "efficiency_0.95": replace(ENV_CONFIG, irrigation_efficiency=0.95),
    "planting_minus_14d": replace(ENV_CONFIG, planting_day_offset=-14),
    "planting_plus_14d": replace(ENV_CONFIG, planting_day_offset=14),
    "kc_0.9": replace(ENV_CONFIG, kc_multiplier=0.9),
    "kc_1.1": replace(ENV_CONFIG, kc_multiplier=1.1),
}

REWARD_SENSITIVITY_CONFIGS = {
    "shortage_weight_5": replace(ENV_CONFIG, shortage_weight=5.0),
    "shortage_weight_20": replace(ENV_CONFIG, shortage_weight=20.0),
}


def make_season_schedule(seed, total_steps):
    # Shared episode schedule; independent from network and exploration RNGs.
    episode_count = int(np.ceil(total_steps / ENV_CONFIG.season_length_days)) + 1
    rng = np.random.default_rng(seed)
    return tuple(int(year) for year in rng.choice(TRAIN_YEARS, size=episode_count, replace=True))


def configuration_fingerprint(config=ENV_CONFIG):
    payload = {"environment": asdict(config), "training": SHARED_TRAINING, "data_sha256": DATA_SHA256}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


EXPERIMENT_TOTALS = {
    "tuning": 3 * 4 * 2 * 30_000,
    "five_seed_final_training": 3 * 5 * 100_000,
    "reward_sensitivity": 3 * 2 * 2 * 30_000,
}
EXPERIMENT_TOTALS["total_learning_interactions"] = sum(EXPERIMENT_TOTALS.values())

print("Configuration fingerprint:", configuration_fingerprint())
print("Planned learning interactions:", EXPERIMENT_TOTALS)


## 6. MDP, observation and simulator equations

At day $t$, the agent receives $O_t$, selects irrigation $I_t$, and the environment returns $R_{t+1}$ and $O_{t+1}$. The hidden simulator state includes storage, day, selected historical season and weather history. The policy does not receive the season identifier or future weather. Therefore, the internal simulator is Markov, but the agent faces a **partially observable** control problem; seven past weather days are an approximation to a sufficient observation [1, Sec. 3.1].

We model grain maize planted on 1 November for 150 days. The single crop-coefficient schedule is 0.30 for days 0–29, increases to 1.20 over days 30–69, remains 1.20 for days 70–119 and decreases to 0.35 over days 120–149. This transfers the FAO-56 example calendar to a southern-hemisphere summer as an explicit assumption rather than a measured Kyabram cultivar [2,10].

The nominal 1.0 m reservoir capacity is 111.3333 mm, calculated from the audited CSIRO AWC layers: 6.25 mm (0–5 cm), 13.00 mm (5–15 cm), 18.25 mm (15–30 cm), 34.50 mm (30–60 cm) and 39.3333 mm (60–100 cm). These are service-generated digital-soil estimates, not field measurements [11,12]. Each episode begins at $0.8C$. The separate 0.6 m sensitivity case uses the first four layers, giving 72.0 mm.

The 18 values are: $A_t/C$, $t/150$, $K_{c,t}/1.2$, $I_{t-1}/20$, seven previous rainfall values divided by 20 and seven previous ETo values divided by 10. The current day's weather is unknown when the action is chosen.

The policy action $u_t\in[-1,1]$ maps to $I_t=10(u_t+1)$ mm. The fixed daily calculation is:

$$ET_{c,t}=K_{c,t}ET_{o,t},\quad p_t=\operatorname{clip}[0.55+0.04(5-ET_{c,t}),0.1,0.8],\quad A_{\min,t}=(1-p_t)C,$$

$$B_t=A_t+P_t+\eta I_t,\quad D_t=\max(0,B_t-C),\quad H_t=B_t-D_t,$$

$$K_{s,t}=\min(1,H_t/A_{\min,t}),\quad ET_{a,t}=\min(H_t,K_{s,t}ET_{c,t}),\quad A_{t+1}=H_t-ET_{a,t}.$$

The mass balance must satisfy $A_{t+1}=A_t+P_t+\eta I_t-D_t-ET_{a,t}$. The model assumes zero runoff and capillary rise, immediate bucket-overflow drainage and no explicit yield or waterlogging damage. These are simplifying assumptions, not measured Kyabram field behaviour [2,3,9,10].

The environmental reward is:

$$R_{t+1}=-\left[\frac{I_t}{20}+10\max\left(0,\frac{A_{\min,t}-A_{t+1}}{A_{\min,t}}\right)+\frac{D_t}{20}\right].$$

The weights $(1,10,1)$ are declared project preferences, not published crop-response parameters. Reward sensitivity therefore repeats short runs with shortage weights 5 and 20. The shared discount is $\gamma=0.995$; SAC's entropy term is excluded from the environmental return used for comparison.


## 7. Shared Gymnasium environment

The action is selected before the current day's rainfall and ETo are revealed. The next observation contains only completed weather history. Diagnostic `info` is used for evaluation and debugging but is never an actor or critic input.


In [ ]:
KYABRAM_MEMBER = 'irrigation_audit/kyabram_080091_daily.json'

class KyabramIrrigationEnv(gym.Env):
    metadata = {'render_modes': ['rgb_array'], 'render_fps': 8}

    def __init__(self, climate_path=DATA_PATH, split='train', config=ENV_CONFIG, render_mode=None):
        super().__init__()
        if render_mode not in (None, "rgb_array"):
            raise ValueError("Supported render_mode: rgb_array or None")
        self.render_mode = render_mode
        if split not in SEASON_SPLITS:
            raise ValueError(f'Unknown split: {split}')
        self.config = config
        self.split = split
        self.allowed_years = SEASON_SPLITS[split]
        records = self._load_records(Path(climate_path))
        self._weather, self._dates = self._build_seasons(records)
        self.action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(
            low=np.zeros(18, dtype=np.float32),
            high=np.array([1.0, 1.0, 1.1, 1.0] + [np.inf] * 14, dtype=np.float32),
            dtype=np.float32,
        )
        self.season_year = None
        self.day = config.season_length_days
        self.storage_mm = 0.0
        self.previous_irrigation_mm = 0.0
        self._terminated = True

    @staticmethod
    def _load_records(path):
        if path.is_dir():
            path = path / 'kyabram_080091_daily.json'
        if path.suffix.lower() == '.zip':
            with ZipFile(path) as archive:
                payload = json.loads(archive.read(KYABRAM_MEMBER))
        else:
            payload = json.loads(path.read_text())
        records = {}
        for index, row in enumerate(payload):
            try:
                current_date = date.fromisoformat(row['date'])
                rain = float(row['rain_mm'])
                eto = float(row['eto_mm'])
            except (KeyError, TypeError, ValueError) as exc:
                raise ValueError(f'Malformed climate row {index}') from exc
            if current_date in records or not np.isfinite([rain, eto]).all():
                raise ValueError(f'Invalid climate input on {current_date}')
            if rain < 0 or eto < 0:
                raise ValueError(f'Negative climate input on {current_date}')
            records[current_date] = (rain, eto)
        return records

    def _build_seasons(self, records):
        weather, dates = {}, {}
        block_length = self.config.history_days + self.config.season_length_days
        for year in SEASON_SPLITS['all']:
            planting = date(year, self.config.planting_month, self.config.planting_day)
            planting += timedelta(days=self.config.planting_day_offset)
            start = planting - timedelta(days=self.config.history_days)
            block_dates = tuple(start + timedelta(days=i) for i in range(block_length))
            missing = [day for day in block_dates if day not in records]
            if missing:
                raise ValueError(f'Season {year} is missing {missing[0]}')
            weather[year] = np.asarray([records[day] for day in block_dates], dtype=np.float64)
            dates[year] = block_dates
        return weather, dates

    def _kc(self, day):
        if day >= self.config.season_length_days:
            return 0.0
        if day < 30:
            base = 0.30
        elif day < 70:
            base = 0.30 + 0.90 * (day - 29) / 40
        elif day < 120:
            base = 1.20
        else:
            base = 1.20 - 0.85 * (day - 119) / 30
        return self.config.kc_multiplier * base

    def _observation(self):
        history = self._weather[self.season_year][self.day:self.day + 7][::-1]
        observation = np.concatenate([
            np.array([
                self.storage_mm / self.config.capacity_mm,
                self.day / self.config.season_length_days,
                self._kc(self.day) / 1.2,
                self.previous_irrigation_mm / self.config.max_irrigation_mm,
            ]),
            history[:, 0] / self.config.rainfall_scale_mm,
            history[:, 1] / self.config.eto_scale_mm,
        ]).astype(np.float32)
        if not np.isfinite(observation).all() or not self.observation_space.contains(observation):
            raise RuntimeError('Invalid observation produced')
        return observation

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        options = {} if options is None else dict(options)
        unknown_options = set(options) - {'season_year'}
        if unknown_options:
            raise ValueError(f'Unknown reset options: {sorted(unknown_options)}')
        requested_year = options.get('season_year')
        if requested_year is None:
            self.season_year = self.allowed_years[int(self.np_random.integers(len(self.allowed_years)))]
        elif int(requested_year) in self.allowed_years:
            self.season_year = int(requested_year)
        else:
            raise ValueError(f'Season {requested_year} is not in the {self.split} split')
        self.day = 0
        self.storage_mm = self.config.initial_storage_fraction * self.config.capacity_mm
        self.previous_irrigation_mm = 0.0
        self._terminated = False
        decision_date = self._dates[self.season_year][7]
        info = {'split': self.split, 'season_year': self.season_year, 'decision_date': decision_date.isoformat()}
        self._visual_info = info.copy()
        self._visual_reward = 0.0
        self._visual_total_irrigation = 0.0
        return self._observation(), info

    def step(self, action):
        if self._terminated:
            raise RuntimeError('Episode ended; call reset()')
        action = np.asarray(action, dtype=np.float32).reshape(-1)
        if action.shape != (1,) or not self.action_space.contains(action):
            raise ValueError('Expected one finite action in [-1, 1]')

        irrigation = 0.5 * self.config.max_irrigation_mm * (float(action[0]) + 1.0)
        weather_index = 7 + self.day
        rain, eto = self._weather[self.season_year][weather_index]
        decision_date = self._dates[self.season_year][weather_index]
        kc = self._kc(self.day)
        potential_et = kc * float(eto)
        depletion = float(np.clip(0.55 + 0.04 * (5.0 - potential_et), 0.1, 0.8))
        minimum = (1.0 - depletion) * self.config.capacity_mm

        storage_before = self.storage_mm
        incoming = storage_before + float(rain) + self.config.irrigation_efficiency * irrigation
        drainage = max(0.0, incoming - self.config.capacity_mm)
        wet_storage = incoming - drainage
        stress_coefficient = min(1.0, wet_storage / minimum)
        actual_et = min(wet_storage, stress_coefficient * potential_et)
        storage_after = wet_storage - actual_et
        shortage = max(0.0, minimum - storage_after)

        irrigation_penalty = self.config.irrigation_weight * irrigation / self.config.max_irrigation_mm
        shortage_penalty = self.config.shortage_weight * shortage / minimum
        drainage_penalty = self.config.drainage_weight * drainage / self.config.max_irrigation_mm
        reward = -(irrigation_penalty + shortage_penalty + drainage_penalty)
        self.storage_mm = storage_after
        self.previous_irrigation_mm = irrigation
        self.day += 1
        self._terminated = self.day == self.config.season_length_days
        residual = storage_after - (storage_before + rain + self.config.irrigation_efficiency * irrigation - drainage - actual_et)
        info = {
            'season_year': self.season_year, 'completed_day': self.day,
            'decision_date': decision_date.isoformat(), 'irrigation_mm': irrigation,
            'rain_mm': float(rain), 'eto_mm': float(eto), 'kc': kc,
            'potential_et_mm': potential_et, 'actual_et_mm': actual_et,
            'depletion_fraction': depletion, 'stress_coefficient': stress_coefficient,
            'minimum_storage_mm': minimum, 'storage_before_mm': storage_before,
            'storage_after_mm': storage_after, 'shortage_mm': shortage,
            'drainage_mm': drainage, 'irrigation_penalty': irrigation_penalty,
            'shortage_penalty': shortage_penalty, 'drainage_penalty': drainage_penalty,
            'balance_residual_mm': residual,
        }
        self._visual_info = info.copy()
        self._visual_reward = float(reward)
        self._visual_total_irrigation += irrigation
        return self._observation(), float(reward), self._terminated, False, info

    def render(self):
        if self.render_mode == "rgb_array":
            return irrigation_rgb_frame(self)
        return None

    def close(self):
        # Each RGB frame uses a short-lived off-screen canvas; no window is retained.
        pass


In [ ]:
# Visualisation pattern: Practical 1 RGB-frame capture and Practical 9 policy replay.
# Matplotlib supplies notebook playback without MoviePy or an external video encoder.
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.figure import Figure
from matplotlib.patches import Rectangle, Ellipse, Circle
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display


def irrigation_rgb_frame(env):
    """Render simulator values only; this function never advances or changes the state."""
    if env.season_year is None:
        raise RuntimeError("Call reset() before render().")
    info = env._visual_info
    fig = Figure(figsize=(9, 5), dpi=80, facecolor="#102238")
    canvas = FigureCanvasAgg(fig)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set(xlim=(0, 100), ylim=(0, 60))
    ax.axis("off")
    ax.add_patch(Rectangle((2, 17), 62, 34, color="#c6e9ee"))
    ax.add_patch(Rectangle((2, 4), 62, 13, color="#805632"))
    ax.add_patch(Rectangle((2, 16), 62, 2, color="#6c9e42"))
    fraction = env.storage_mm / env.config.capacity_mm
    ax.add_patch(Rectangle((2, 4), 62, 12 * fraction, color="#359ec5", alpha=.8))
    ax.add_patch(Circle((56, 44), 3, color="#ffd261"))
    progress = env.day / env.config.season_length_days
    crop_color = "#39843d" if progress < .8 else "#a59b35"
    # Illustrative crop height follows calendar progress, not a simulated yield model.
    height = 5 + 17 * min(progress / .6, 1)
    for x in range(9, 62, 9):
        ax.plot([x, x], [18, 18 + height], color=crop_color, lw=3)
        for side, level in [(-1, .35), (1, .55), (-1, .75)]:
            ax.add_patch(Ellipse((x + side * 1.5, 18 + height * level),
                                 5, 1.7, angle=side * 30, color=crop_color))
        for dx in [-2, 0, 2]:
            ax.plot([x, x + dx], [17, 10], color="#e4cf9e", lw=1)
    rain = info.get("rain_mm", 0.)
    irrigation = info.get("irrigation_mm", 0.)
    if rain > 0:
        for x in range(8, 45, 5):
            ax.plot([x, x-1], [46, 43], color="#367fca", lw=2)
        ax.text(6, 49, "RAINFALL", fontsize=9, color="#234b73")
    ax.add_patch(Rectangle((3, 19), 3, 7, color="#738494"))
    ax.plot([4, 8], [26, 26], color="#738494", lw=4)
    if irrigation > 0:
        for end in [17, 23, 29]:
            ax.plot([8, 12, end], [26, 29, 19], color="#198fdf", lw=1.5, ls="--")
    minimum = info.get("minimum_storage_mm")
    if minimum is not None:
        y = 4 + 12 * minimum / env.config.capacity_mm
        ax.plot([2, 64], [y, y], "--", color="#fff0a1", lw=1.5)
    ax.text(4, 5, "ROOT-ZONE WATER (schematic)", color="white", fontsize=9)
    ax.text(3, 55, "KYABRAM  |  SMART IRRIGATION", color="white", fontsize=15, weight="bold")
    ax.text(67, 49, f"DAY {env.day:03d} / {env.config.season_length_days}",
            color="#a7dfe6", fontsize=13, weight="bold")
    date_label = info.get("decision_date", "")
    rows = [date_label, f"Soil water: {env.storage_mm:.1f} mm",
            f"Capacity: {env.config.capacity_mm:.1f} mm",
            f"Irrigation: {irrigation:.1f} mm",
            f"Rainfall: {rain:.1f} mm",
            f"Crop ET: {info.get('actual_et_mm', 0.):.1f} mm",
            f"Drainage: {info.get('drainage_mm', 0.):.1f} mm",
            f"Reward: {env._visual_reward:.3f}",
            f"Total irrigation: {env._visual_total_irrigation:.1f} mm"]
    for i, line in enumerate(rows):
        ax.text(67, 44 - i*4, line, color="white", fontsize=9)
    status = "INITIAL STATE" if minimum is None else (
        "BELOW WATER TARGET" if env.storage_mm < minimum else "WITHIN WATER TARGET")
    ax.text(3, 1, status + "  |  Crop artwork is illustrative; no yield is modelled.",
            color="#ffcf85", fontsize=8)
    canvas.draw()
    return np.asarray(canvas.buffer_rgba())[:, :, :3].copy()


def visualise_season(action_function, *, label="Threshold controller", split="validation",
                     season_year=2017, fps=8):
    """Replay one fixed season with learning disabled; return playback and daily values."""
    env = KyabramIrrigationEnv(split=split, render_mode="rgb_array")
    frames, rows = [], []
    try:
        observation, _ = env.reset(seed=11, options={"season_year": season_year})
        frames.append(env.render())
        while True:
            action = np.asarray(action_function(observation), dtype=np.float32).reshape(1)
            observation, reward, terminated, truncated, info = env.step(action)
            frames.append(env.render())
            rows.append({**info, "reward": reward})
            if terminated or truncated:
                break
    finally:
        env.close()
    fig, ax = plt.subplots(figsize=(9, 5))
    fig.subplots_adjust(0, 0, 1, .94)
    ax.axis("off")
    ax.set_title(f"{label} | {split} season {season_year}", fontsize=10)
    picture = ax.imshow(frames[0])
    def update(index):
        picture.set_data(frames[index])
        return (picture,)
    animation = FuncAnimation(fig, update, frames=len(frames), interval=1000/fps, blit=True)
    html = animation.to_jshtml(fps=fps, embed_frames=True, default_mode="once")
    plt.close(fig)
    return HTML(html), pd.DataFrame(rows)


## 8. Environment validation and problem diagnosis

These tests must pass before agent training. They check software consistency, not agronomic validity. The infinite observation-space warning is expected because weather is finite-checked but not clipped using test maxima.


In [ ]:
def expect_exception(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type:
        return
    raise AssertionError(f"Expected {exception_type.__name__}")


checked_env = KyabramIrrigationEnv(split="train")
with warnings.catch_warnings(record=True) as checker_warnings:
    warnings.simplefilter("always")
    check_env(checked_env, skip_render_check=True)
assert any("infinity" in str(item.message).lower() for item in checker_warnings)

# Reproducible reset and valid initial observation.
first_observation, first_info = checked_env.reset(seed=11)
second_observation, second_info = checked_env.reset(seed=11)
np.testing.assert_array_equal(first_observation, second_observation)
assert first_info["season_year"] == second_info["season_year"]
assert checked_env.observation_space.contains(first_observation)
np.testing.assert_allclose(first_observation[:4], [0.8, 0.0, 0.25, 0.0])

# Split enforcement and option validation.
for split, year in (("train", 2005), ("validation", 2017), ("test", 2021)):
    split_env = KyabramIrrigationEnv(split=split)
    _, info = split_env.reset(options={"season_year": year})
    assert info["season_year"] == year
expect_exception(ValueError, checked_env.reset, options={"season_year": 2021})
expect_exception(ValueError, checked_env.reset, options={"unknown": 1})

# Action mapping and no future-weather leakage.
initial, _ = checked_env.reset(options={"season_year": 2005})
next_observation, _, _, _, step_info = checked_env.step(np.array([-1.0], dtype=np.float32))
assert step_info["irrigation_mm"] == 0.0
np.testing.assert_allclose(next_observation[4] * ENV_CONFIG.rainfall_scale_mm, step_info["rain_mm"])
np.testing.assert_allclose(next_observation[11] * ENV_CONFIG.eto_scale_mm, step_info["eto_mm"])
np.testing.assert_allclose(next_observation[5:11], initial[4:10])
np.testing.assert_allclose(next_observation[12:18], initial[11:17])

checked_env.reset(options={"season_year": 2005})
_, _, _, _, maximum_info = checked_env.step(np.array([1.0], dtype=np.float32))
assert maximum_info["irrigation_mm"] == ENV_CONFIG.max_irrigation_mm
for invalid_action in (
    np.array([-1.01], dtype=np.float32), np.array([1.01], dtype=np.float32),
    np.array([np.nan], dtype=np.float32), np.array([0.0, 0.0], dtype=np.float32),
):
    checked_env.reset(options={"season_year": 2005})
    expect_exception(ValueError, checked_env.step, invalid_action)

# Hand-check dry, irrigation-only and heavy-rain/near-capacity cases.
def controlled_case(storage_mm, rain_mm, eto_mm, action):
    case_env = KyabramIrrigationEnv(split="train")
    case_env.reset(options={"season_year": 2005})
    case_env._weather[2005] = case_env._weather[2005].copy()
    case_env._weather[2005][ENV_CONFIG.history_days] = (rain_mm, eto_mm)
    case_env.storage_mm = storage_mm
    return case_env.step(np.array([action], dtype=np.float32))

_, _, _, _, dry = controlled_case(20.0, 0.0, 5.0, -1.0)
assert dry["storage_after_mm"] < dry["storage_before_mm"] and dry["drainage_mm"] == 0.0

_, _, _, _, irrigation_only = controlled_case(50.0, 0.0, 0.0, 1.0)
np.testing.assert_allclose(irrigation_only["storage_after_mm"], 50.0 + 0.85 * 20.0)

_, _, _, _, heavy_rain = controlled_case(0.95 * ENV_CONFIG.capacity_mm, 50.0, 0.0, -1.0)
assert heavy_rain["drainage_mm"] > 0.0
np.testing.assert_allclose(heavy_rain["storage_after_mm"], ENV_CONFIG.capacity_mm)

# Complete random episode, exact mass balance and terminal handling.
episode_env = KyabramIrrigationEnv(split="train")
observation, _ = episode_env.reset(seed=11, options={"season_year": 2005})
rng = np.random.default_rng(11)
max_balance_error = 0.0
for step_index in range(150):
    action = rng.uniform(-1, 1, size=1).astype(np.float32)
    observation, reward, terminated, truncated, info = episode_env.step(action)
    assert episode_env.observation_space.contains(observation)
    assert np.isfinite(reward) and reward <= 0.0 and not truncated
    max_balance_error = max(max_balance_error, abs(info["balance_residual_mm"]))
    assert terminated is (step_index == 149)
assert max_balance_error < 1e-10
assert observation[1] == 1.0 and observation[2] == 0.0
expect_exception(RuntimeError, episode_env.step, np.array([0.0], dtype=np.float32))

VALIDATION_REPORT = {
    "gymnasium_checker": "passed with expected infinite-bound warning",
    "reproducible_reset": True,
    "weather_timing": "passed",
    "controlled_cases": "passed",
    "terminal_day": 150,
    "max_abs_balance_residual_mm": max_balance_error,
}
VALIDATION_REPORT


### Environment timing and feasibility

The environment performs a fixed amount of arithmetic and no external calls per step. The quick benchmark below verifies the active runtime. Before final training, change to 100,000 steps and five repetitions, then separately measure each agent's collection, gradient-update and four-season evaluation time. Environment-only timing is not an end-to-end training estimate.


In [ ]:
def benchmark_environment(steps=10_000, repetitions=3, seed=6008):
    actions = np.random.default_rng(seed).uniform(-1, 1, size=(steps, 1)).astype(np.float32)
    elapsed = []
    for repetition in range(repetitions):
        timing_env = KyabramIrrigationEnv(split="train")
        timing_env.reset(seed=seed + repetition)
        started = time.perf_counter()
        for action in actions:
            _, _, terminated, _, _ = timing_env.step(action)
            if terminated:
                timing_env.reset()
        elapsed.append(time.perf_counter() - started)
    return {
        "steps_per_repetition": steps,
        "repetitions": repetitions,
        "elapsed_seconds": elapsed,
        "median_microseconds_per_step": 1e6 * float(np.median(elapsed)) / steps,
        "runtime": RUNTIME,
    }


ENVIRONMENT_TIMING = benchmark_environment()
ENVIRONMENT_TIMING


## 9. Shared utilities adapted from the practicals

The replay buffer and target-network pattern adapt Practical 8. Actor/critic separation and MLP construction adapt Practical 9. Continuous actions, critic inputs and algorithm-specific losses are new project work, as documented in [`PRACTICAL_CODE_REUSE.md`](../docs/PRACTICAL_CODE_REUSE.md).


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class ReplayBuffer:
    def __init__(self, capacity=SHARED_TRAINING['replay_capacity']):
        self.memory = deque(maxlen=capacity)

    def add(self, observation, action, reward, next_observation, terminated):
        self.memory.append((observation, action, reward, next_observation, terminated))

    def sample(self, batch_size=SHARED_TRAINING['batch_size']):
        batch = random.sample(self.memory, batch_size)
        observations, actions, rewards, next_observations, terminals = map(np.asarray, zip(*batch))
        return tuple(torch.as_tensor(value, dtype=torch.float32, device=DEVICE) for value in (
            observations, actions, rewards[:, None], next_observations, terminals[:, None]
        ))

    def __len__(self):
        return len(self.memory)

def build_mlp(input_size, output_size, output_activation=None):
    h1, h2 = SHARED_TRAINING['hidden_sizes']
    layers = [nn.Linear(input_size, h1), nn.ReLU(), nn.Linear(h1, h2), nn.ReLU(), nn.Linear(h2, output_size)]
    if output_activation is not None:
        layers.append(output_activation())
    return nn.Sequential(*layers)

def adam(parameters, learning_rate):
    return torch.optim.Adam(
        parameters, lr=learning_rate, betas=SHARED_TRAINING['optimiser_betas'],
        eps=SHARED_TRAINING['optimiser_epsilon'],
    )


def soft_update(target, source, tau=SHARED_TRAINING['tau']):
    with torch.no_grad():
        for target_parameter, source_parameter in zip(target.parameters(), source.parameters()):
            target_parameter.mul_(1.0 - tau).add_(source_parameter, alpha=tau)


## 10. Shared benchmark and evaluation functions

All policies are evaluated on complete seasons with learning disabled. Physical outcomes accompany reward so a policy cannot be described as better merely because it exploits a trade-off in the scalar reward.


In [ ]:
def zero_irrigation_policy(observation):
    return np.array([-1.0], dtype=np.float32)


def maximum_irrigation_policy(observation):
    return np.array([1.0], dtype=np.float32)


def threshold_policy(observation, capacity=ENV_CONFIG.capacity_mm):
    storage = float(observation[0]) * capacity
    if storage < 0.65 * capacity:
        irrigation = min(20.0, (0.90 * capacity - storage) / 0.85)
    else:
        irrigation = 0.0
    return np.array([irrigation / 10.0 - 1.0], dtype=np.float32)


def evaluate_controller(action_function, split="validation", config=ENV_CONFIG,
                        label="policy", seed=None, training_steps=0):
    evaluation_env = KyabramIrrigationEnv(split=split, config=config)
    rows = []
    for year in SEASON_SPLITS[split]:
        observation, _ = evaluation_env.reset(options={"season_year": year})
        discounted_return = undiscounted_return = nominal_rescored_return = 0.0
        irrigation_total = shortage_total = drainage_total = 0.0
        actual_et_total = potential_et_total = 0.0
        days_below_target = 0
        for day in range(config.season_length_days):
            action = np.asarray(action_function(observation), dtype=np.float32).reshape(1)
            observation, reward, terminated, truncated, info = evaluation_env.step(action)
            discounted_return += (SHARED_TRAINING["gamma"] ** day) * reward
            undiscounted_return += reward
            nominal_reward = -(
                info["irrigation_mm"] / config.max_irrigation_mm
                + 10.0 * info["shortage_mm"] / info["minimum_storage_mm"]
                + info["drainage_mm"] / config.max_irrigation_mm
            )
            nominal_rescored_return += nominal_reward
            irrigation_total += info["irrigation_mm"]
            shortage_total += info["shortage_mm"]
            drainage_total += info["drainage_mm"]
            actual_et_total += info["actual_et_mm"]
            potential_et_total += info["potential_et_mm"]
            days_below_target += int(info["shortage_mm"] > 0.0)
        assert terminated and not truncated
        rows.append({
            "agent": label, "seed": seed, "configuration": None,
            "split": split, "season_year": year, "training_steps": training_steps,
            "discounted_return": discounted_return,
            "undiscounted_return": undiscounted_return,
            "nominal_rescored_return": nominal_rescored_return,
            "irrigation_mm": irrigation_total,
            "days_below_target": days_below_target,
            "shortage_mm_days": shortage_total,
            "actual_to_potential_et_ratio": actual_et_total / potential_et_total,
            "drainage_mm": drainage_total,
            "status": "ok",
        })
    return pd.DataFrame(rows)


def evaluate_agent(agent, split="validation", config=ENV_CONFIG, label="agent", seed=None, training_steps=0):
    return evaluate_controller(
        lambda observation: agent.select_action(observation, evaluate=True),
        split=split, config=config, label=label, seed=seed, training_steps=training_steps,
    )


benchmark_results = pd.concat([
    evaluate_controller(zero_irrigation_policy, label="zero"),
    evaluate_controller(maximum_irrigation_policy, label="maximum"),
    evaluate_controller(threshold_policy, label="threshold"),
], ignore_index=True)
benchmark_results.groupby("agent").mean(numeric_only=True)


### Visual farm replay — following Practicals 1 and 9

Use the play/pause and frame controls below to watch one daily decision at a time.
As in Practical 1, `render_mode="rgb_array"` enables RGB-frame capture with
`env.render()`. As in Practical 9, replay uses a policy with learning disabled.
Matplotlib's HTML player replaces MoviePy so playback needs no external encoder.
See [PRACTICAL_CODE_REUSE.md](../docs/PRACTICAL_CODE_REUSE.md) for the wider
practical-code provenance.

The water layer, rainfall, sprinkler activity, drainage and reward reflect the actual
simulator. Crop artwork follows season progress and is illustrative, not a yield
prediction. The dashed line is that completed day's minimum soil-water target.
Weather shown is revealed **after** the action, never supplied early to the agent.
Rendering is off during training. Playback is a demonstration, not evidence of
performance across seeds/seasons. The template demo uses the threshold controller;
agent notebooks can pass their own deterministic evaluation policy to the same helper.


In [ ]:
threshold_playback, threshold_daily = visualise_season(threshold_policy)
display(threshold_playback)


Season replay: threshold controller, validation 2017

## 11. DDPG — member work area

**Owner:** Raina Shah  
**Practical starting structure:** Practicals 8 and 9  
**Original method:** Lillicrap et al. [4]

Required components: bounded deterministic actor; action-conditioned critic; replay; actor/critic targets; Gaussian training exploration clipped to $[-1,1]$; critic TD target; deterministic policy-gradient update; soft target updates; deterministic evaluation.

| Pseudocode step | Code method/lines | Practical element retained | Continuous-control change | Evidence |
|---|---|---|---|---|
| TODO | TODO | TODO | TODO | [4] |


In [ ]:
# DDPG IMPLEMENTATION PLACEHOLDER
# Selected only after validation tuning:
DDPG_LEARNING_RATE = AGENT_SETTINGS['ddpg']['selected_learning_rate']
DDPG_EXPLORATION_STD = AGENT_SETTINGS['ddpg']['selected_exploration_std']

class DDPGAgent:
    # TODO: implement using the Practical 9 actor-critic structure and off-policy replay ideas.
    pass

## 12. TD3 — member work area

**Owner:** Arnav Dangmali  
**Practical starting structure:** Practicals 8 and 9  
**Original method:** Fujimoto et al. [5]

Required components: bounded deterministic actor; two independent critics; minimum target value; clipped target-policy smoothing; delayed actor and target updates; bounded Gaussian exploration; deterministic evaluation.

| Pseudocode step | Code method/lines | Practical element retained | TD3-specific change | Evidence |
|---|---|---|---|---|
| TODO | TODO | TODO | TODO | [5] |


In [ ]:
# TD3 IMPLEMENTATION PLACEHOLDER
TD3_LEARNING_RATE = AGENT_SETTINGS['td3']['selected_learning_rate']
TD3_EXPLORATION_STD = AGENT_SETTINGS['td3']['selected_exploration_std']
TD3_POLICY_DELAY = AGENT_SETTINGS['td3']['policy_delay']
TD3_TARGET_SMOOTHING_STD = AGENT_SETTINGS['td3']['target_smoothing_std']
TD3_TARGET_NOISE_CLIP = AGENT_SETTINGS['td3']['target_noise_clip']

class TD3Agent:
    # TODO: implement twin critics, minimum target and delayed policy updates.
    pass

## 13. SAC — member work area

**Owner:** Malaika Noor  
**Practical starting structure:** Practical 9  
**Original method:** Haarnoja et al. [6]

Implement the agreed 2018 formulation: tanh-squashed Gaussian policy, reparameterised sampling, tanh log-probability correction, two Q-functions, separate value and target-value networks, fixed $\alpha$, soft target update and deterministic evaluation using tanh of the Gaussian mean.

| Pseudocode step | Code method/lines | Practical element retained | SAC-specific change | Evidence |
|---|---|---|---|---|
| TODO | TODO | TODO | TODO | [6] |


In [ ]:
# SAC IMPLEMENTATION PLACEHOLDER
SAC_LEARNING_RATE = AGENT_SETTINGS['sac']['selected_learning_rate']
SAC_ALPHA = AGENT_SETTINGS['sac']['selected_alpha']
SAC_LOG_STD_BOUNDS = AGENT_SETTINGS['sac']['log_std_bounds']

class SACAgent:
    # TODO: implement the agreed 2018 SAC formulation with a separate value network.
    pass

## 14. Shared agent interface and training loop

Every agent must provide `select_action(observation, evaluate=False, rng=None)`, `update(replay_buffer)` and `save(path)`. The loop below uses a shared season schedule but separate schedule, network and exploration random streams. Only genuine day-150 termination removes bootstrap value. Reaching the external interaction budget does not fabricate a terminal transition.


In [ ]:
RESULT_COLUMNS = [
    "agent", "configuration", "seed", "split", "season_year", "training_steps",
    "discounted_return", "undiscounted_return", "nominal_rescored_return",
    "irrigation_mm", "days_below_target", "shortage_mm_days",
    "actual_to_potential_et_ratio", "drainage_mm", "runtime_seconds", "status",
]


def train_agent(agent, seed, total_steps, agent_name, config=ENV_CONFIG):
    set_seed(seed)  # network/global stream
    exploration_rng = np.random.default_rng(seed + 10_000)
    season_schedule = make_season_schedule(seed, total_steps)
    training_env = KyabramIrrigationEnv(split="train", config=config)
    replay_buffer = ReplayBuffer()
    episode_index = 0
    observation, _ = training_env.reset(options={"season_year": season_schedule[episode_index]})
    evaluation_rows = []
    update_rows = []
    started = time.perf_counter()

    # Step-zero evaluation is required by the fixed schedule.
    evaluation_rows.append(evaluate_agent(
        agent, label=agent_name, seed=seed, training_steps=0, config=config
    ))

    for training_step in range(1, total_steps + 1):
        if training_step <= SHARED_TRAINING["random_exploration_steps"]:
            action = exploration_rng.uniform(-1, 1, size=1).astype(np.float32)
        else:
            action = np.asarray(
                agent.select_action(observation, evaluate=False, rng=exploration_rng),
                dtype=np.float32,
            ).reshape(1)
            action = np.clip(action, -1.0, 1.0).astype(np.float32)

        next_observation, reward, terminated, truncated, info = training_env.step(action)
        assert not truncated
        replay_buffer.add(observation, action, reward, next_observation, terminated)
        observation = next_observation

        if training_step > SHARED_TRAINING["random_exploration_steps"]:
            for _ in range(SHARED_TRAINING["updates_per_environment_step"]):
                diagnostics = agent.update(replay_buffer)
                if diagnostics:
                    update_rows.append({"training_steps": training_step, **diagnostics})

        if terminated:
            episode_index += 1
            observation, _ = training_env.reset(
                options={"season_year": season_schedule[episode_index]}
            )

        if training_step % SHARED_TRAINING["evaluation_interval"] == 0:
            evaluation_rows.append(evaluate_agent(
                agent, label=agent_name, seed=seed,
                training_steps=training_step, config=config,
            ))

    runtime_seconds = time.perf_counter() - started
    evaluations = pd.concat(evaluation_rows, ignore_index=True)
    evaluations["runtime_seconds"] = runtime_seconds
    return {
        "evaluations": evaluations,
        "updates": pd.DataFrame(update_rows),
        "season_schedule": season_schedule[:episode_index + 1],
        "runtime_seconds": runtime_seconds,
        "replay_buffer": replay_buffer,
    }


## 15. Tuning, checkpoint selection and experiment accounting

Each agent receives four configurations × seeds 101 and 202 × 30,000 transitions. Evaluate all four validation seasons at step 30,000. Choose the highest mean discounted environmental return; break an exact tie by lower irrigation and then declared configuration order. Any configuration with a failed run receives $-\infty$ and remains recorded. The final test uses the fixed step-100,000 checkpoint—not the most favourable intermediate checkpoint.


In [ ]:
TUNING_COLUMNS = [
    "agent", "configuration_index", "configuration", "seed", "season_year",
    "training_steps", "discounted_return", "irrigation_mm", "status", "notes",
]
tuning_record = pd.DataFrame(columns=TUNING_COLUMNS)


def tuning_plan():
    rows = []
    for agent_name, configurations in AGENT_TUNING_GRIDS.items():
        for configuration_index, configuration in enumerate(configurations):
            for seed in SHARED_TRAINING["tuning_seeds"]:
                rows.append({
                    "agent": agent_name,
                    "configuration_index": configuration_index,
                    "configuration": configuration,
                    "seed": seed,
                    "training_steps": SHARED_TRAINING["tuning_steps"],
                })
    return pd.DataFrame(rows)


def select_tuning_configuration(records, agent_name):
    required = set(TUNING_COLUMNS) - {"notes"}
    missing = required - set(records.columns)
    if missing:
        raise ValueError(f"Missing tuning columns: {sorted(missing)}")
    subset = records[records["agent"] == agent_name].copy()
    expected_seasons = set(VALIDATION_YEARS)
    summaries = []
    for configuration_index in range(len(AGENT_TUNING_GRIDS[agent_name])):
        group = subset[subset["configuration_index"] == configuration_index]
        expected_rows = len(SHARED_TRAINING["tuning_seeds"]) * len(VALIDATION_YEARS)
        complete = len(group) == expected_rows and set(group["season_year"]) == expected_seasons
        valid = complete and group["status"].eq("ok").all() and np.isfinite(group["discounted_return"]).all()
        summaries.append({
            "configuration_index": configuration_index,
            "configuration": AGENT_TUNING_GRIDS[agent_name][configuration_index],
            "selection_score": float(group["discounted_return"].mean()) if valid else -np.inf,
            "mean_irrigation_mm": float(group["irrigation_mm"].mean()) if valid else np.inf,
            "valid": valid,
        })
    summary = pd.DataFrame(summaries).sort_values(
        ["selection_score", "mean_irrigation_mm", "configuration_index"],
        ascending=[False, True, True],
    )
    if not np.isfinite(summary.iloc[0]["selection_score"]):
        raise RuntimeError(f"Every {agent_name} configuration failed; repair and version before restarting")
    return summary.iloc[0].to_dict(), summary


print(tuning_plan())


## 16. Stability, sample efficiency and held-out summaries

“Stable” is an operational plateau label, not a convergence proof. The range of the last five mean validation checkpoints must be at most $0.05\max(1,|\bar G|)$. Sample efficiency is the first checkpoint that reaches the threshold benchmark and remains at or above it for the next two checkpoints. Report “not reached” otherwise.

For final testing, first average the four test seasons within each seed. Then report the mean, sample standard deviation and all five seed values. Do not treat days or seed–season pairs as independent replicates.


In [ ]:
def mean_learning_curve(evaluations):
    return (
        evaluations.groupby("training_steps", as_index=False)["discounted_return"]
        .mean().sort_values("training_steps")
    )


def operational_stability(evaluations):
    curve = mean_learning_curve(evaluations)
    if len(curve) < 5:
        return {"stable": False, "reason": "fewer than five checkpoints"}
    values = curve.tail(5)["discounted_return"].to_numpy(dtype=float)
    mean_return = float(values.mean())
    observed_range = float(values.max() - values.min())
    tolerance = 0.05 * max(1.0, abs(mean_return))
    return {
        "stable": observed_range <= tolerance,
        "last_five_range": observed_range,
        "tolerance": tolerance,
        "mean_return": mean_return,
    }


def sample_efficiency(evaluations, threshold_benchmark_return):
    curve = mean_learning_curve(evaluations)
    steps = curve["training_steps"].to_numpy()
    reached = curve["discounted_return"].to_numpy() >= threshold_benchmark_return
    for index in range(len(reached) - 2):
        if reached[index:index + 3].all():
            return int(steps[index])
    return "not reached"


def summarise_final_test(test_results):
    seed_level = (
        test_results.groupby(["agent", "seed"], as_index=False)
        .agg({
            "discounted_return": "mean", "undiscounted_return": "mean",
            "irrigation_mm": "mean", "days_below_target": "mean",
            "shortage_mm_days": "mean", "actual_to_potential_et_ratio": "mean",
            "drainage_mm": "mean",
        })
    )
    overall = seed_level.groupby("agent").agg(["mean", "std"])
    return seed_level, overall


## 17. Reward sensitivity and robustness/generalisation

After configurations are frozen:

- train short reward-sensitivity runs with shortage weights 5 and 20 using seeds 101 and 202, then compare physical outcomes and nominally rescored returns;
- evaluate each final frozen policy without retraining on all four test seasons under nine one-at-a-time changes: $0.8C$, $1.2C$, 0.6 m root zone, efficiencies 0.75/0.95, planting ±14 days and $K_c$ ×0.9/×1.1; and
- retain the nominal 20 mm action limit and 150-day season.

These tests examine selected assumptions, not generalisation to every farm or climate. Capacity scenarios use known recalibration in the observation. Efficiency remains hidden from the RL policy. The changed $K_c$ is visible through the existing crop-coefficient observation.


In [ ]:
def reward_sensitivity_plan():
    rows = []
    for agent_name in AGENT_TUNING_GRIDS:
        for scenario_name in REWARD_SENSITIVITY_CONFIGS:
            for seed in SHARED_TRAINING["tuning_seeds"]:
                rows.append({
                    "agent": agent_name, "scenario": scenario_name, "seed": seed,
                    "training_steps": SHARED_TRAINING["tuning_steps"],
                    "split": "train/validation only",
                })
    return pd.DataFrame(rows)


def evaluate_robustness(agent, agent_name, seed):
    rows = []
    for scenario_name, scenario_config in ROBUSTNESS_CONFIGS.items():
        scenario_results = evaluate_agent(
            agent, split="test", config=scenario_config,
            label=agent_name, seed=seed,
            training_steps=SHARED_TRAINING["final_checkpoint"],
        )
        scenario_results["scenario"] = scenario_name
        rows.append(scenario_results)
    return pd.concat(rows, ignore_index=True)


def evaluate_robustness_benchmark():
    rows = []
    for scenario_name, scenario_config in ROBUSTNESS_CONFIGS.items():
        # The threshold controller receives scenario capacity but retains its nominal 0.85 refill equation.
        results = evaluate_controller(
            lambda observation, capacity=scenario_config.capacity_mm: threshold_policy(observation, capacity),
            split="test", config=scenario_config, label="threshold",
            training_steps=0,
        )
        results["scenario"] = scenario_name
        rows.append(results)
    return pd.concat(rows, ignore_index=True)


print(reward_sensitivity_plan())


## 18. Failure analysis and debugging record

Do not attribute a failed run to an algorithm until the environment, reward, observations, exploration, gradients and numerical updates have been checked. Failed and unsuccessful runs remain part of the evidence.


In [ ]:
FAILURE_COLUMNS = [
    "date", "agent", "configuration", "seed", "training_step", "observation",
    "suspected_cause", "evidence_checked", "change_made", "outcome",
    "resolved", "reviewer",
]
failure_log = pd.DataFrame(columns=FAILURE_COLUMNS)

INTEGRATION_CHECKLIST = pd.DataFrame([
    {"check": "Agent accepts 18-value float32 observation", "status": "TODO", "reviewer": None},
    {"check": "Training action is finite and bounded", "status": "TODO", "reviewer": None},
    {"check": "Terminal target has no bootstrap value", "status": "TODO", "reviewer": None},
    {"check": "External budget cut-off retains bootstrap", "status": "TODO", "reviewer": None},
    {"check": "Evaluation disables exploration/learning", "status": "TODO", "reviewer": None},
    {"check": "Weights, results and runtime persist outside memory", "status": "TODO", "reviewer": None},
    {"check": "Pseudocode maps to implementation", "status": "TODO", "reviewer": None},
])
INTEGRATION_CHECKLIST


## 19. Results, interpretation and communication template

### Results to insert

1. Tuning table including failed trials and the selected configuration for each agent.
2. Validation learning curves against environment interactions, showing all seeds or uncertainty bands.
3. Final test results by season and by seed, plus five-seed mean and sample SD.
4. Physical trade-offs: irrigation, dry days, cumulative shortage, actual/potential ET and drainage.
5. Stability and sample-efficiency outcomes using the predeclared rules.
6. Nine robustness scenarios and reward-sensitivity results.
7. Agent-specific and end-to-end runtime measurements.

### Interpretation discipline

For every major conclusion, write:

- **Evidence:** the precise result or cited source.
- **Reasoning:** why it supports the conclusion.
- **Alternative explanation:** another plausible cause.
- **Assumptions and limits:** what was not established.

Do not call a noisy plateau convergence. Do not call lower irrigation an improvement if shortage rises materially. Held-out years at one location do not establish transfer to other farms. Simulated water status is not measured yield or plant health.

### Improvements and future extensions

| Member/agent | Demonstrated limitation | Evidence from results | Proposed improvement | Scientific source | Separate future direction |
|---|---|---|---|---|---|
| DDPG owner | TODO | TODO | TODO | TODO | TODO |
| TD3 owner | TODO | TODO | TODO | TODO | TODO |
| SAC owner | TODO | TODO | TODO | TODO | TODO |


In [ ]:
# Load saved CSV/JSON result files here; do not rely only on notebook memory.
all_results = pd.DataFrame(columns=RESULT_COLUMNS)

# Required figures after runs exist:
# 1. Mean validation return versus environment interactions, with seed variation.
# 2. Test irrigation versus shortage trade-off.
# 3. Test metrics by agent and season.
# 4. Robustness metric changes relative to the nominal scenario.


## 20. Feasibility and milestone record

The environment is feasible because it uses in-memory arrays and explicit arithmetic; no paid service, crop-growth package, solver or live hardware is required. This does not guarantee that every neural agent will train successfully.

| Date | Milestone | Completion evidence | Status |
|---|---|---|---|
| 5–6 October | Data, settings and shared environment | Complete seasons, fixed interface and configuration fingerprint | Complete |
| 6–7 October | Physics and information-timing validation | Controlled cases, mass balance, reproducibility and terminal checks | Complete in template; peer review TODO |
| 7 October | Agent integration and environment timing | Three smoke runs plus active-runtime timing | TODO |
| 7–8 October | Independent environment review | Equations/code/settings checked by another member | TODO |
| 11 October | Agent implementations and runtime pilots | DDPG, TD3 and SAC integrated; 10,000-step timings | TODO |
| 15 October | Tuning | All declared configurations, seeds and failures recorded | TODO |
| 18 October | Final experiments | Five-seed tests, reward sensitivity and robustness saved | TODO |
| 19–22 October | Analysis and presentation | Reproducible figures/tables and recorded presentation | TODO |
| 29 October | Report review | Group review before 30 October deadline | TODO |

If runtime becomes restrictive, first reduce avoidable I/O and logging. Any budget change must be documented and applied equally before final testing; test performance cannot be used to give a preferred agent extra tuning.


## 21. References

[1] R. S. Sutton and A. G. Barto, *Reinforcement Learning: An Introduction*, 2nd ed., MIT Press, 2018, ISBN 9780262039246.

[2] R. G. Allen, L. S. Pereira, D. Raes, and M. Smith, *Crop Evapotranspiration: Guidelines for Computing Crop Water Requirements*, FAO Irrigation and Drainage Paper 56, 1998.

[3] K. L. Greenwood, A. R. Lawson, and K. B. Kelly, “The water balance of irrigated forages in northern Victoria, Australia,” *Agricultural Water Management*, 96(5), 847–858, 2009. https://doi.org/10.1016/j.agwat.2008.11.005

[4] T. P. Lillicrap et al., “Continuous control with deep reinforcement learning,” *ICLR*, 2016. https://arxiv.org/abs/1509.02971

[5] S. Fujimoto, H. van Hoof, and D. Meger, “Addressing function approximation error in actor-critic methods,” *ICML/PMLR 80*, 1587–1596, 2018. https://proceedings.mlr.press/v80/fujimoto18a.html

[6] T. Haarnoja, A. Zhou, P. Abbeel, and S. Levine, “Soft actor-critic: Off-policy maximum entropy deep reinforcement learning with a stochastic actor,” *ICML/PMLR 80*, 1861–1870, 2018. https://proceedings.mlr.press/v80/haarnoja18b.html

[7] P. Henderson et al., “Deep reinforcement learning that matters,” *AAAI*, 32(1), 2018. https://doi.org/10.1609/aaai.v32i1.11694

[8] S. J. Jeffrey et al., “Using spatial interpolation to construct a comprehensive archive of Australian climate data,” *Environmental Modelling & Software*, 16(4), 309–330, 2001. https://doi.org/10.1016/S1364-8152(01)00008-1

[9] K. L. Greenwood, G. N. Mundy, and K. B. Kelly, “On-farm measurement of the water use and productivity of maize,” *Australian Journal of Experimental Agriculture*, 48(3), 274–284, 2008. https://doi.org/10.1071/EA06094

[10] L. S. Pereira et al., “Standard single and basal crop coefficients for field crops,” *Agricultural Water Management*, 243, 106466, 2021. https://doi.org/10.1016/j.agwat.2020.106466

[11] B. P. Malone et al., “Update and expansion of the soil and landscape grid of Australia,” *Geoderma*, 455, 117226, 2025. https://doi.org/10.1016/j.geoderma.2025.117226

[12] R. Searle, P. D. S. N. Somarathna, and B. Malone, *Soil and Landscape Grid National Soil Attribute Maps – Available Volumetric Water Capacity*, CSIRO dataset, 2022. https://doi.org/10.25919/4jwj-na34

Publication status, DOI checks and claim limits are documented in [`PROJECT_OBJECTIVES.md`](../docs/PROJECT_OBJECTIVES.md). The textbook, FAO manual and dataset are authoritative or provenance sources rather than peer-reviewed journal papers; the original algorithm papers are peer-reviewed conference papers without identified conference DOIs.


## 22. Final reproducibility checklist

- [ ] All three agents use this environment and the same saved configuration fingerprint.
- [ ] Environment validation passes on the final runtime and receives independent review.
- [ ] Practical reuse and every algorithm's pseudocode-to-code mapping are documented.
- [ ] Agent-specific `None` values are filled from validation tuning only.
- [ ] All tuning configurations and failed runs are retained.
- [ ] Test seasons are untouched until environment, reward, settings and selection are frozen.
- [ ] All five final seeds use 100,000 interactions and the same season schedule per seed.
- [ ] Step-zero and 5,000-step validation evaluations contain no learning or replay insertion.
- [ ] True terminal transitions do not bootstrap; external cut-offs do.
- [ ] Step 100,000—not a favourable intermediate checkpoint—is used for final tests.
- [ ] Seed-level and season-level results, sample SD, learning curves and physical metrics are reported.
- [ ] Stability and sample efficiency use the declared operational definitions.
- [ ] Reward-sensitivity and all nine one-at-a-time robustness scenarios are completed without test-driven redesign.
- [ ] Weights, CSV/JSON results, package versions, hardware and runtimes are saved persistently.
- [ ] Conclusions separate evidence, reasoning, alternatives and limitations.
- [ ] Every member can explain the environment and all three agents.
